# 04. Leakage Audit

Repeats the leakage checks used throughout this project against the final feature set and split, and adds a direct contrast against a naive random row split, the concrete leakage the client grouped design closes.

## Setup

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')")
con.execute("SET threads=1")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# the five behavioral features the final KMeans model clusters on, shared by every
# notebook downstream of the model fit so the feature set cannot silently drift between
# the clustering, validation, and leakage audit notebooks
FEATURE_COLS = ["ctr", "avg_position", "days_since_last_update", "total_impressions_log", "engagement_rate_filled"]

page_level = pd.read_csv("../interim/page_level.csv")
model_data = pd.read_csv("../interim/model_data.csv")
train = pd.read_csv("../interim/train.csv")
test = pd.read_csv("../interim/test.csv")

## Rebuild a naive split for contrast

model_data gets split by row here instead of by client, the same naive split used in the split comparison notebook. The audit needs a concrete number to contrast against the grouped split's zero client overlap, not just an assertion that grouping matters. naive_train and naive_test get rebuilt here independently.

In [3]:
naive_train, naive_test = train_test_split(model_data, test_size=0.2, random_state=42)

## The audit

Same checks that were run on the rule based baseline earlier in this project get repeated here against this model's final feature set and split. A model can look correct and still be leaking information from the future, from a client, or from a label that should never have been a feature, so running the same checklist again after every meaningful change is cheap insurance against that. The staleness reference date stays inside the fact table's own reporting window, zero rows have a negative days_since_last_update, none of FEATURE_COLS come from a product flag or a precomputed tier, the grouped split has zero client overlap while the naive split has dozens, which is the concrete leakage the grouped split closes, and none of archetype, cannibalization_risk, or keyword_hash_id ended up inside FEATURE_COLS.

In [4]:
# 1. staleness reference date is bounded by the fact table's own max date
print(con.sql(f"""
    SELECT MAX(report_date) AS max_report_date, MIN(report_date) AS min_report_date
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df())

# 2. negative days_since_last_update fix held in the final modeling population
print(f"negative days_since_last_update rows in model_data: {(model_data['days_since_last_update'] < 0).sum()}")

# 3. no product flag or precomputed tier used as a clustering input
used_features = set(FEATURE_COLS)
fact_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')").df()["column_name"].tolist()
dim_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet')").df()["column_name"].tolist()
print("Fact columns NOT in FEATURE_COLS:", [c for c in fact_cols if c not in used_features])
print("Dim columns NOT in FEATURE_COLS:", [c for c in dim_cols if c not in used_features])

# 4. client split integrity, grouped split vs the naive split rebuilt above
overlap_clients_check = set(train["client_hash_id"]) & set(test["client_hash_id"])
naive_overlap_check = set(naive_train["client_hash_id"]) & set(naive_test["client_hash_id"])
print(f"GROUPED split clients appearing in both train and test: {len(overlap_clients_check)}")
print(f"NAIVE split clients appearing in both train and test: {len(naive_overlap_check)}, "
      f"this is the leakage the grouped split closes")

# 5. downstream labels not accidentally used as clustering inputs
print("archetype in FEATURE_COLS:", "archetype" in FEATURE_COLS)
print("cannibalization_risk in FEATURE_COLS:", "cannibalization_risk" in FEATURE_COLS)
print("keyword_hash_id in FEATURE_COLS:", "keyword_hash_id" in FEATURE_COLS)

  max_report_date min_report_date
0      2026-06-30      2026-06-01
negative days_since_last_update rows in model_data: 0


Fact columns NOT in FEATURE_COLS: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
Dim columns NOT in FEATURE_COLS: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_opti

## One snapshot, and what gets excluded from it before any split happens

This entire project is built from one data pull covering June 2026, one reporting window, so the future window leakage a multi period project has to watch for, a feature aggregated over a span that overlaps the label's own span, has no way to occur here, there is only one span. A time based split is not applicable for the same reason, there is no second period to hold out. That is a property of the data, not a check that passed, and it is worth saying plainly rather than silently skipping the question.

The population itself is a choice worth naming too. model_data excludes every page the rule engine marked INSUFFICIENT_DATA, no GSC data or fewer than 10 impressions, roughly 69 percent of the full page level table. That filter runs on the same impressions and GSC availability fields the clustering features are later built from, so it is not neutral, a page too small to measure never gets a chance to be clustered or archetyped at all. That is a disclosed scope limit, already named in the accompanying paper's limitations section, not a leak, but it belongs in this audit's line of sight too since it is exactly the kind of population question a leakage review should ask.

FEATURE_COLS excludes every product flag and existing system decision in the warehouse, not just the columns that happen to differ from the rule engine's own inputs. provider_used, model_used, and optimization_eligible_date in particular are decisions a prior system already made about a page, using them as clustering inputs would mean partly learning that system's old choices back, not the page's own behavior, so they get named and checked here explicitly rather than left inside the general "columns not in FEATURE_COLS" list above.

In [5]:
# decision-derived signals: flags or scores a prior system already produced, not raw page
# behavior. confirming these specific ones stay out of FEATURE_COLS, named individually
# rather than folded into the generic column dump in the audit above
product_flag_cols = ["provider_used", "model_used", "optimization_eligible_date", "is_published", "is_deleted"]
print("decision-derived columns confirmed absent from FEATURE_COLS:")
print([c for c in product_flag_cols if c not in FEATURE_COLS])

insufficient_data_rows = (page_level["archetype"] == "INSUFFICIENT_DATA").sum()
print(f"\nrows excluded from model_data by the INSUFFICIENT_DATA filter: {insufficient_data_rows} of {len(page_level)}, "
      f"{insufficient_data_rows / len(page_level) * 100:.1f}%")

decision-derived columns confirmed absent from FEATURE_COLS:
['provider_used', 'model_used', 'optimization_eligible_date', 'is_published', 'is_deleted']

rows excluded from model_data by the INSUFFICIENT_DATA filter: 272549 of 394928, 69.0%


## Attacking the harness: does this audit actually notice a leaky feature

Every check above confirms FEATURE_COLS looks clean, none of that proves the audit would catch it if it were not. The honest way to check a test is to deliberately fail it, twice, with two different kinds of failure. ctr_gap gets added to FEATURE_COLS first, KMeans gets refit at the same k=20, and the resulting clusters get scored against the rule based archetype with the same Adjusted Rand Index used everywhere else in this project, right next to the honest score using the real, unmodified FEATURE_COLS.

In [6]:
train_leak_check = train.merge(
    model_data[["client_hash_id", "content_hash_id", "ctr_gap"]],
    on=["client_hash_id", "content_hash_id"], how="left",
)

honest_clusters = train_leak_check["kmeans_cluster"].values

scaler_leaky = RobustScaler()
X_leaky = scaler_leaky.fit_transform(train_leak_check[FEATURE_COLS + ["ctr_gap"]])
kmeans_leaky = KMeans(n_clusters=20, random_state=42, n_init=10)
leaky_clusters = kmeans_leaky.fit_predict(X_leaky)

mask = (train_leak_check["archetype"] != "cannibalization_risk").values
rule_labels = train_leak_check["archetype"].values

ari_honest = adjusted_rand_score(rule_labels[mask], honest_clusters[mask])
ari_leaky = adjusted_rand_score(rule_labels[mask], leaky_clusters[mask])

print(f"ARI vs rule based archetype, honest FEATURE_COLS (no ctr_gap): {ari_honest:.3f}")
print(f"ARI vs rule based archetype, FEATURE_COLS plus ctr_gap injected: {ari_leaky:.3f}")
print("ctr_gap is not part of FEATURE_COLS in the real model, this run exists only to confirm the audit would catch it if it were")

ARI vs rule based archetype, honest FEATURE_COLS (no ctr_gap): 0.098
ARI vs rule based archetype, FEATURE_COLS plus ctr_gap injected: 0.110
ctr_gap is not part of FEATURE_COLS in the real model, this run exists only to confirm the audit would catch it if it were


The ARI barely moves, 0.098 to 0.110. Read plainly that could mean the audit is not sensitive to leakage, which is a real risk worth checking rather than explaining away, so it gets checked with a second, stronger injection right below. But there is a more specific reason to expect exactly this outcome first: ctr_gap is expected_ctr minus ctr, and expected_ctr comes from avg_position through a four bucket lookup, so ctr_gap is close to a fixed transform of ctr and avg_position, both of which are already in FEATURE_COLS. Adding it barely changes the 5 dimensional space KMeans already sees, it is closer to duplicating an axis than adding one, so a small ARI move here does not yet say whether the audit would catch a feature that carries information the honest model genuinely does not have.

A label encoded version of archetype itself is the sharper test, information the honest model has no path to, not a transform of what it already sees. This is the direct case the leakage taxonomy warns about, the label riding along inside the features, not a milder variant of it.

In [7]:
train_leak_check["archetype_label_encoded"] = train_leak_check["archetype"].astype("category").cat.codes

scaler_direct_leak = RobustScaler()
X_direct_leak = scaler_direct_leak.fit_transform(train_leak_check[FEATURE_COLS + ["archetype_label_encoded"]])
kmeans_direct_leak = KMeans(n_clusters=20, random_state=42, n_init=10)
direct_leak_clusters = kmeans_direct_leak.fit_predict(X_direct_leak)

ari_direct_leak = adjusted_rand_score(rule_labels[mask], direct_leak_clusters[mask])

print(f"ARI vs rule based archetype, honest FEATURE_COLS: {ari_honest:.3f}")
print(f"ARI vs rule based archetype, FEATURE_COLS plus ctr_gap injected: {ari_leaky:.3f}")
print(f"ARI vs rule based archetype, FEATURE_COLS plus archetype itself injected: {ari_direct_leak:.3f}")

ARI vs rule based archetype, honest FEATURE_COLS: 0.098
ARI vs rule based archetype, FEATURE_COLS plus ctr_gap injected: 0.110
ARI vs rule based archetype, FEATURE_COLS plus archetype itself injected: 0.357


0.357 against a 0.098 honest baseline is the real confession the ctr_gap test could not produce, more than three times the honest agreement, from adding exactly one column. That confirms the harness does react when genuine label information is present, the earlier flat result was ctr_gap carrying little information beyond what ctr and avg_position already provide, not the audit being blind to leakage. It is also worth being precise about what 0.357 is not, it is nowhere near 1.0, because the injected column is one of six dimensions in a distance based clustering competing with five honest features, not the sole input, and because k=20 splits the data into far more groups than the 8 rule based labels, which caps how high agreement with an 8 category label can climb even with the label sitting directly inside the features. A smaller, real jump reads as leakage just as much as a jump to 1.0 would, expecting only the extreme case would miss this one.

## What this audit does and does not claim

No sealed or blind holdout is claimed anywhere in this project. The test split gets used openly throughout, the clustering notebook's own train and test silhouette, the split comparison notebook, and every check in this notebook, so there is no sealed claim here that would need a separately committed frame builder and metrics file, that requirement is not applicable to a test split that gets reused openly like this one rather than touched once and sealed.

Test cluster assignments in train.csv and test.csv come from calling predict on the clustering notebook's already fit KMeans, not from fitting a second model on test, so every test side number reported anywhere in this project is out of fold, test rows never influenced the centroids they get scored against.

The F-statistic ranking computed on the model side already sanity checks what the clustering leans on, two close leading features well ahead of the rest, not one implausibly dominant column. That is the same shape a clean feature set should produce, and it is the opposite of what the ctr_gap injection above produces on purpose, which is the point of running both.